# 04 – Monte Carlo DCF valuation

A classic DCF gives **one** number that hides how uncertain it is. Here we simulate 10,000 plausible futures
and look at the **distribution** of intrinsic value per share.

**Model (per path)**
* Revenue grows at `g_high` for *N* years, then fades linearly to terminal growth by year 10
* Free cash flow = revenue × margin
* Value = PV of 10 years of FCF + PV of terminal value (Gordon growth) + net cash, divided by diluted shares

**Random inputs:** high-phase growth, margin, WACC, terminal growth (normal distributions).
Defaults come from the company's last 5 fiscal years (median growth, mean margin, their volatility).

⚠️ **Read before trusting a number**
* The defaults **extrapolate the past**. Hyper-growth is capped at 30 % but even that may be too optimistic.
* **Financials** (banks, insurers, Berkshire): FCF is meaningless → DCF is the wrong tool, they are ranked last.
* **Heavy investment phase** (Amazon, Walmart, Oracle…): FCF margin is temporarily depressed → try *net margin* basis.
* Net cash = cash − long-term debt only (short-term debt, investments, leases ignored).
* Corporate actions (spin-offs like GE) distort historical growth.

In [ ]:
import pandas as pd
from IPython.display import display

from stockpred.config import load_settings
from stockpred.prices import last_close
from stockpred.valuation_mc import estimate_inputs, simulate, sensitivity_grid, screen_universe
from stockpred.valuation_ui import build_valuation_widget
from stockpred.plots import mc_histogram_figure, sensitivity_heatmap

settings = load_settings()
panel = pd.read_parquet(settings.processed_dir / "fundamentals_annual.parquet")
universe = pd.read_csv(settings.processed_dir / "universe.csv")

# fresh prices (one batch request); fall back to the prices stored in notebook 01
prices = last_close(universe["ticker"].tolist()).combine_first(universe.set_index("ticker")["price"])
print(f"Prices for {prices.notna().sum()} / {len(universe)} companies")

## 1. One company, step by step

In [ ]:
TICKER = "MSFT"
row = universe.set_index("ticker").loc[TICKER]
inputs, assumptions = estimate_inputs(panel, TICKER, prices[TICKER], row["sub_industry"])
print(inputs)
print(assumptions)

result = simulate(inputs, assumptions)
print("Percentiles:", {k: round(v, 1) for k, v in result.percentiles().items()})
print(f"P(value > price) = {result.prob_undervalued:.0%}, median vs price = {result.margin_of_safety:+.0%}")
for w in result.warnings:
    print("⚠️", w)
mc_histogram_figure(result, TICKER).show()

**Sensitivity** – the same DCF without randomness, over a grid of WACC × growth.
It shows which assumption the value really depends on (usually WACC and terminal value).

In [ ]:
sensitivity_heatmap(sensitivity_grid(inputs, assumptions), inputs.price).show()

## 2. Interactive – change the assumptions yourself

Pick a company, move the sliders (the chart updates when you release the mouse), switch the margin basis,
or *Reset to history*. The random seed is fixed, so moving one slider only changes that input.

In [ ]:
build_valuation_widget(panel, universe, prices)

## 3. Screen the whole universe with default assumptions

⚠️ This is a **mechanical** screen with history-based defaults – a list of companies worth a closer look in the
interactive widget above, not a buy list. Financial companies are listed last.

In [ ]:
screen = screen_universe(panel, universe, prices, margin_basis="fcf")
screen.to_csv(settings.processed_dir / "mc_screen.csv", index=False)
cols = ["ticker", "name", "sector", "dcf_applicable", "price", "value_p5", "value_median", "value_p95",
        "margin_of_safety", "prob_undervalued", "growth_assumed", "fcf_margin_assumed"]
display(screen[cols].style.format({
    "price": "{:,.2f}", "value_p5": "{:,.0f}", "value_median": "{:,.0f}", "value_p95": "{:,.0f}",
    "margin_of_safety": "{:+.0%}", "prob_undervalued": "{:.0%}",
    "growth_assumed": "{:.1%}", "fcf_margin_assumed": "{:.1%}",
}).background_gradient(subset=["prob_undervalued"], cmap="RdYlGn", vmin=0, vmax=1))

### How much do the results depend on WACC?
Re-run the screen with a different discount rate for all companies – if the ranking changes a lot,
the "signal" is mostly an assumption.

In [ ]:
for wacc in (0.08, 0.09, 0.10):
    s = screen_universe(panel, universe, prices, wacc_mean=wacc)
    top = s[s["dcf_applicable"]].head(5)["ticker"].tolist()
    print(f"WACC {wacc:.0%}: median P(undervalued) {s['prob_undervalued'].median():.0%}, top 5: {top}")